### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [1]:
# =====================================================================
# STEP 1: IMPORT REQUIRED LANGCHAIN CORE COMPONENTS & UTILITIES
# =====================================================================

# Factory function used to initialize specialized chat models from different providers (e.g., Groq, OpenAI)
from langchain.chat_models import init_chat_model

# Formatter template to construct structured prompts with placeholders for the language model
from langchain_core.prompts import PromptTemplate

# Document loader used to extract raw text content from local files
from langchain_community.document_loaders import TextLoader

# Document processing utility that splits long texts into smaller chunks using character constraints
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding wrapper used to transform raw text into dense vector configurations using HuggingFace models
from langchain_openai import OpenAIEmbeddings

# Vector database wrapper managing semantic indexing and high-speed similarity search using FAISS
from langchain_community.vectorstores import FAISS

# Standard output parser that converts a model's rich ChatMessage response directly into a clean text string
from langchain_core.output_parsers import StrOutputParser

# Utility function that packs a list of fetched documents directly into a single language model context window
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Composition wrapper utilized to chain sequence items sequentially in LangChain Expression Language (LCEL)
from langchain_core.runnables import RunnableSequence

C:\Users\shiva\AppData\Local\Temp\ipykernel_16840\2247057208.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# =====================================================================
# STEP 2: LOAD KNOWLEDGE DATASET, SPLIT INTO CHUNKS, AND CREATE RETRIEVER
# =====================================================================

# 2.1. Instantiate the loader targeting your local unstructured text document
loader = TextLoader("langchain_crewai_dataset.txt")

# 2.2. Extract raw contents from the file into structural LangChain Document formats
docs = loader.load()

# 2.3. Define text chunk partitioning parameters to handle semantic boundaries and overlap
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# 2.4. Apply chunking rules to segment the documents into micro-contexts
chunks = splitter.split_documents(docs)

# 2.5. Select the sentence transformer model used to project textual semantic meaning into embeddings
embedding = OpenAIEmbeddings()

# 2.6. Build and populate an in-memory FAISS vector index using the generated chunks and embeddings
vectorstore = FAISS.from_documents(chunks, embedding)

# 2.7. Expose the vector database as a document retriever using Maximal Marginal Relevance (MMR)
# Adjusts relevance (k) and diversity parameters (lambda_mult) to minimize informational redundancy
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

In [4]:
# =====================================================================
# STEP 3: ENVIRONMENT MANAGEMENT AND CHAT MODEL CONFIGURATION
# =====================================================================

# Import standard operational tools to query runtime system environment attributes
import os

# Import tool to inspect local .env configurations and automatically apply keys to runtime execution
from dotenv import load_dotenv

# Extract matching records from file and populate local environment parameters
load_dotenv()

# Assign the API credential variable to tell downstream LangChain functions where to authentic with OpenAI
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# Initialize the chat model utilizing the unified init_chat_model framework targeting the 'o4-mini' engine
llm=init_chat_model("openai:o4-mini")

# Return instance details to confirm configuration attributes
llm

ChatOpenAI(output_version=None, profile={'name': 'o4-mini', 'release_date': '2025-04-16', 'last_updated': '2025-04-16', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001AC7FDBC090>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AC7FDBF210>, root_client=<openai.OpenAI object at 0x000001AC7FDBD790>, root_async_client=<openai.AsyncOpenAI object at 0x000001AC7FDBE590>, model_name='o4-mini', model_kwargs={}, openai_

In [5]:
# =====================================================================
# STEP 4: DEFINE PROMPT TEMPLATE AND CHAIN FOR QUERY DECOMPOSITION
# =====================================================================

# 4.1. Define strict context guidelines directing the model to slice complex inputs into atomic topics
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

# 4.2. Chain prompt creation, model execution, and string parsing sequentially using pipe syntax (LCEL)
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [6]:
# =====================================================================
# STEP 5: PROMPT SIMULATION RUN - EVALUATING QUERY BREAKDOWN STRATEGY
# =====================================================================

# 5.1. Declare a complex user question involving multiple distinct comparative ideas
query = "How does LangChain use memory and agents compared to CrewAI?"

# 5.2. Send payload to decomposition pipeline to trigger itemization into distinct sub-questions
decomposition_question=decomposition_chain.invoke({"question": query})

In [7]:
# =====================================================================
# DISPLAY GENERATED SUB-QUESTIONS
# =====================================================================

# Print out the raw multi-line string generated by the decomposition sequence
print(decomposition_question)

Here are four focused sub-questions to guide your document search:

1. What memory modules and storage strategies does LangChain provide for maintaining conversational context?  
2. How does CrewAI implement and manage memory compared to LangChain?  
3. How are agents defined, orchestrated, and invoked within LangChain’s framework?  
4. What agent model and orchestration mechanisms does CrewAI use, and how do they differ from LangChain’s approach?


In [8]:
# =====================================================================
# STEP 6: CONSTRUCT INTERNAL DOCUMENT CONTEXT QUESTION-ANSWERING CHAIN
# =====================================================================

# 6.1. Establish a prompt layout formatting document context lists right beside user question inputs
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

# 6.2. Bind the language engine and prompt formatting into a unified context-stuffing pipeline handler
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [9]:
# =====================================================================
# STEP 7: BUILD COMPREHENSIVE QUERY DECOMPOSITION RAG PROCESSING ENGINE
# =====================================================================

def full_query_decomposition_rag_pipeline(user_query):
    # 7.1. Process primary question text to produce a single aggregated string of sub-questions
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    
    # 7.2. Parse and sanitize output: split by line breaks, strip numbering/bullets, ignore empty lines
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    # 7.3. Loop through every single parsed itemized atomic sub-question sequentially
    results = []
    for subq in sub_questions:
        # A. Query vector database retriever using the current localized sub-question
        docs = retriever.invoke(subq)
        
        # B. Inject individual contexts and inputs directly into the document response processor
        result = qa_chain.invoke({"input": subq, "context": docs})
        
        # C. Format result pairs cleanly and collect them inside an tracking output collection array
        results.append(f"Q: {subq}\nA: {result}")
    
    # 7.4. Merge entire collected response items using double-newlines and return complete text log
    return "\n\n".join(results)

In [10]:
# =====================================================================
# STEP 8: PIPELINE RUNTIME EXECUTION AND SYSTEM OUTPUT DISPLAY
# =====================================================================

# 8.1. Define original multi-concept query statement for structural validation
query = "How does LangChain use memory and agents compared to CrewAI?"

# 8.2. Execute the overarching query decomposition engine loop against the knowledge store documents
final_answer = full_query_decomposition_rag_pipeline(query)

# 8.3. Output tracking headers to execution logs
print("✅ Final Answer:\n")

# 8.4. Render complete concatenated sub-question contexts and model text logs into the cell console
print(final_answer)

✅ Final Answer:

Q: Here are four targeted sub-questions to guide your retrieval:
A: Here are four targeted sub-questions you can use to guide your retrieval process:

1. Which retrieval methods does LangChain support out of the box (e.g. BM25, vector embeddings, etc.)?  
2. How is hybrid retrieval implemented in LangChain, and what benefits does it deliver over pure sparse or pure dense search?  
3. In what form and at what point in the pipeline is fetched knowledge injected into the LLM prompt?  
4. What concrete impact does prompt-level knowledge injection have on model accuracy and hallucination rates?

Q: What memory-management features and abstractions does LangChain provide?
A: LangChain comes with a built-in “memory” abstraction and a suite of ready-made memory modules so that your chains and agents can keep track of what’s happened so far without you having to roll your own state-management.  Out of the box you get, for example:

 • ConversationBufferMemory  
   – the simplest